In [2]:
import pandas as pd
import numpy as np
import time
from pathlib import Path

print("Avvio Modulo 02: Data Ingestion & Vettorial Merge...\n")
start_time = time.time()

# --- 1. CONFIGURAZIONE PERCORSI (CROSS-PLATFORM) ---
current_dir = Path.cwd()
if current_dir.name == 'notebooks':
    BASE_DIR = current_dir.parent
else:
    BASE_DIR = current_dir

DATA_DIR_RAW = BASE_DIR / "data" / "raw"
DATA_DIR_PROCESSED = BASE_DIR / "data" / "processed"

# Assicuriamoci che la cartella di destinazione esista
DATA_DIR_PROCESSED.mkdir(parents=True, exist_ok=True)

path_es_03 = DATA_DIR_RAW / "ES 03-26.Last.txt"
path_es_06 = DATA_DIR_RAW / "ES 06-26.Last.txt"
path_cfd = DATA_DIR_RAW / "fpmarkets_us500_ticks.csv"
path_output = DATA_DIR_PROCESSED / "es_master_4months.parquet"

# --- 2. GESTIONE FUSI ORARI E ALLINEAMENTO ---
# IMPORTANTE: Se NinjaTrader era impostato sul tuo orologio locale italiano, 
# cambia 'US/Central' con 'Europe/Rome'. Altrimenti lascia l'orario di Chicago.
nt_timezone = 'US/Central'

# Il file CFD inizia il 9 Febbraio 2026. Tagliamo il Future in eccesso.
data_inizio_cut = pd.to_datetime('2026-02-09 01:00:00').tz_localize('UTC').tz_convert(nt_timezone).tz_localize(None)

# --- 3. CARICAMENTO E PRE-PROCESSING FUTURE ---
def processa_future(file_path, start_date=None):
    print(f"Lettura e calcolo Delta per: {file_path.name}")
    df = pd.read_csv(
        file_path, sep=';', 
        names=['RawTime', 'Price', 'Bid', 'Ask', 'Volume'],
        usecols=[0, 1, 2, 3, 4], engine='c'
    )
    
    # Conversione temporale ottimizzata
    df['CleanTime'] = df['RawTime'].str.slice(0, 15)
    df['Datetime'] = pd.to_datetime(df['CleanTime'], format='%Y%m%d %H%M%S')
    
    # Taglio chirurgico per risparmiare memoria (applicato solo al file di Marzo)
    if start_date:
        df = df[df['Datetime'] >= start_date]
        
    # Conversione in UTC Assoluto per il merge
    df['Datetime_UTC'] = df['Datetime'].dt.tz_localize(nt_timezone).dt.tz_convert('UTC')
    
    # Calcolo del Delta Istantaneo (Tick Rule avanzata)
    df['Direction'] = np.where(df['Price'] >= df['Ask'], 1, np.where(df['Price'] <= df['Bid'], -1, 0))
    df['Delta'] = df['Volume'] * df['Direction']
    
    # Pulizia colonne: teniamo solo l'essenziale per il quant trading
    df = df[['Datetime_UTC', 'Price', 'Volume', 'Delta']].sort_values('Datetime_UTC')
    return df

# Controllo di sicurezza prima di ingolfare la RAM
if not path_es_03.exists() or not path_es_06.exists() or not path_cfd.exists():
    print(f"ERRORE CRITICO: Verifica che i tre file esistano nella cartella:\n{DATA_DIR_RAW}")
else:
    df_03 = processa_future(path_es_03, start_date=data_inizio_cut)
    df_06 = processa_future(path_es_06)

    print("Concatenazione contratti Future...")
    df_future = pd.concat([df_03, df_06], ignore_index=True)
    del df_03, df_06 # Svuotamento forzato della RAM

    # --- 4. CARICAMENTO DATI CFD ---
    print("Lettura storico CFD FP Markets...")
    df_cfd = pd.read_csv(path_cfd, sep=';', usecols=['datetime_utc', 'bid', 'ask'], engine='c')
    df_cfd['Datetime_UTC'] = pd.to_datetime(df_cfd['datetime_utc'], format='ISO8601')
    df_cfd = df_cfd[['Datetime_UTC', 'bid', 'ask']].sort_values('Datetime_UTC')

    # --- 5. IL MERGE VETTORIALE (ALLINEAMENTO MILLISECONDI) ---
    print("Inizio Fusione Asincrona (Merge Asof)...")
    # Abbiniamo ogni tick del Future al prezzo Bid/Ask del CFD più vicino nel futuro
    df_master = pd.merge_asof(
        df_future, 
        df_cfd, 
        on='Datetime_UTC', 
        direction='forward',
        tolerance=pd.Timedelta(seconds=2) # Evita abbinamenti errati se ci sono buchi
    )

    # --- 6. SALVATAGGIO COMPRESSO ---
    print(f"Salvataggio del Master Dataset in formato Parquet: {path_output.name}")
    df_master.to_parquet(path_output, index=False)

    end_time = time.time()
    print(f"\n--- FUSIONE COMPLETATA IN {(end_time - start_time):.2f} SECONDI ---")
    print(f"Righe Totali Master: {len(df_master):,}")
    print("Il dataset è pronto per lo sviluppo dell'algoritmo.")

Avvio Modulo 02: Data Ingestion & Vettorial Merge...

Lettura e calcolo Delta per: ES 03-26.Last.txt
Lettura e calcolo Delta per: ES 06-26.Last.txt
Concatenazione contratti Future...
Lettura storico CFD FP Markets...
Inizio Fusione Asincrona (Merge Asof)...
Salvataggio del Master Dataset in formato Parquet: es_master_4months.parquet


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

In [4]:
print("Tentativo di salvataggio con fastparquet...")
df_master.to_parquet(path_output, engine='fastparquet', index=False)
print("Salvataggio completato!")

Tentativo di salvataggio con fastparquet...
Salvataggio completato!


In [1]:
import pandas as pd
from pathlib import Path

# --- 1. CONFIGURAZIONE PERCORSI (CROSS-PLATFORM) ---
current_dir = Path.cwd()
if current_dir.name == 'notebooks':
    BASE_DIR = current_dir.parent
else:
    BASE_DIR = current_dir

path_output = BASE_DIR / "data" / "processed" / "es_master_4months.parquet"

# --- 2. CARICAMENTO DATI ---
print(f"Caricamento istantaneo da: {path_output.name} ...")
df = pd.read_parquet(path_output)

print(f"\nRighe totali: {len(df):,}")
display(df.head())
display(df.tail())

Caricamento istantaneo da: es_master_4months.parquet ...

Righe totali: 97,712,528


,Datetime_UTC,Price,Volume,Delta,bid,ask
0,2026-02-09 05:00:00+00:00,6960.25,280,280,6944.82,6945.07
1,2026-02-09 05:00:00+00:00,6962.75,1,1,6944.82,6945.07
2,2026-02-09 05:00:00+00:00,6963.00,3,3,6944.82,6945.07
3,2026-02-09 05:00:00+00:00,6963.00,1,1,6944.82,6945.07
4,2026-02-09 05:00:00+00:00,6963.00,1,1,6944.82,6945.07


,Datetime_UTC,Price,Volume,Delta,bid,ask
97712523,2026-06-10 19:44:09+00:00,7384.25,1,-1,NaN,NaN
97712524,2026-06-10 19:44:09+00:00,7384.50,1,1,NaN,NaN
97712525,2026-06-10 19:44:09+00:00,7384.50,2,-2,NaN,NaN
97712526,2026-06-10 19:44:09+00:00,7384.25,1,-1,NaN,NaN
97712527,2026-06-10 19:44:09+00:00,7384.25,2,-2,NaN,NaN
